# NER Pipeline — HuggingFace Transformers + GPU

Model: `Jean-Baptiste/roberta-large-ner-english`  
Good for noisy/historical newspaper text. Handles PERSON, ORG, LOC/GPE.

Swap to `dslim/bert-large-NER` if you want a lighter model with slightly faster throughput.

In [1]:
import os
import json
import logging
import torch
import pandas as pd
import re

from collections import Counter
from pathlib import Path
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

C:\Users\johna\miniforge3\envs\textmining2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Device setup ──────────────────────────────────────────────────────────────
# Priority: CUDA (Nvidia) > XPU (Intel Arc/Data Center) > CPU
if torch.cuda.is_available():
    DEVICE = 0          # HuggingFace pipeline uses integer device index for CUDA
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif hasattr(torch, 'xpu') and torch.xpu.is_available():
    # Intel XPU — pipeline doesn't accept 'xpu' string directly,
    # so we move the model manually after loading (see model cell below).
    DEVICE = 'xpu'
    DEVICE_NAME = 'Intel XPU'
else:
    DEVICE = -1         # CPU fallback
    DEVICE_NAME = 'CPU'

print(f'PyTorch version : {torch.__version__}')
print(f'Selected device : {DEVICE_NAME}')

PyTorch version : 2.7.0+xpu
Selected device : Intel XPU


In [3]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [4]:
# ── Configuration ─────────────────────────────────────────────────────────────
TARGET_DIR  = 'E:/Callproject/extracted_json'

# Use a local path if you have the weights cached, otherwise the HuggingFace
# Hub name below will download on first run.
MODEL_NAME  = 'Jean-Baptiste/roberta-large-ner-english'

# How many texts to feed the model at once.
# Increase if you have >8 GB VRAM; decrease if you hit OOM errors.
BATCH_SIZE  = 32

# Aggregation strategy: 'simple' merges sub-word tokens back into words.
# 'first' and 'max' are alternatives if you see odd splits.
AGGREGATION = 'simple'

In [20]:
# ── Model loading ─────────────────────────────────────────────────────────────
logging.info(f'Loading model: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model     = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
model.eval()

if DEVICE == 'xpu':
    # Intel extension must be imported before moving the model
    import intel_extension_for_pytorch as ipex
    model = model.to('xpu')
    # Wrap with IPEX for optimised inference
    model = ipex.optimize(model, dtype=torch.float32)
    # HuggingFace pipeline needs a plain device string when passing a pre-loaded model
    pipeline_device = 'xpu'
    print(f'Intel PyTorch Extension version: {ipex.__version__}')
else:
    pipeline_device = DEVICE   # int (0 for CUDA) or -1 (CPU)

ner_pipe = pipeline(
    'ner',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy=AGGREGATION,
    device=pipeline_device,
)

logging.info('Pipeline ready.')

INFO: Loading model: Jean-Baptiste/roberta-large-ner-english
2026-04-08 17:37:18,557 - _logger.py - IPEX - INFO - Currently split master weight for xpu only support sgd
2026-04-08 17:37:18,563 - _logger.py - IPEX - INFO - Conv BatchNorm folding failed during the optimize process.
2026-04-08 17:37:18,570 - _logger.py - IPEX - INFO - Linear BatchNorm folding failed during the optimize process.
Device set to use xpu
INFO: Pipeline ready.


Intel PyTorch Extension version: 2.7.10+xpu


In [24]:
# ── Label mapping ─────────────────────────────────────────────────────────────
# roberta-large-ner uses B-/I- prefixed labels.
# Map them to the three categories we care about.
LABEL_MAP = {
    'PER':  'PERSON',
    'ORG':  'ORG',
    'LOC':  'GPE',
    'MISC': None,      # drop miscellaneous
}

def map_label(raw_label: str):
    """Strips B-/I- prefix and maps to our category names."""
    tag = raw_label.split('-')[-1]   # 'B-PER' → 'PER', 'PER' → 'PER'
    return LABEL_MAP.get(tag)


# ── File helpers ──────────────────────────────────────────────────────────────
def get_file_list(root_path, limit=None):
    """Collect JSON paths; apply optional cap for testing."""
    print('Scanning directory for files...')
    files = list(Path(root_path).rglob('*.json'))
    if limit:
        print(f'Test mode: limiting to first {limit} files.')
        return files[:limit]
    return files


def extract_text(file_path) -> str:
    """Pull all paragraph text from one JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as fh:
            data = json.load(fh)
        parts = [
            para['text']
            for page  in data.get('pages',  [])
            for frame in page.get('frames', [])
            for para  in frame.get('paragraphs', [])
            if para.get('text')
        ]
        return ' '.join(parts)
    except Exception:
        return ''


def chunk_text(text: str, max_chars: int = 450) -> list[str]:
    """
    Split on sentence boundaries first, then by word count if a sentence
    is still too long. This prevents named entities like 'Kansas City, Mo.'
    from being torn apart across chunk boundaries.
    """
    # Split on sentence-ending punctuation followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    
    chunks, current, length = [], [], 0
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        # If adding this sentence would overflow, flush current buffer
        if length + len(sent) + 1 > max_chars and current:
            chunks.append(' '.join(current))
            current, length = [], 0
        # If a single sentence is itself too long, split it on word boundaries
        if len(sent) > max_chars:
            words = sent.split()
            for word in words:
                if length + len(word) + 1 > max_chars and current:
                    chunks.append(' '.join(current))
                    current, length = [], 0
                current.append(word)
                length += len(word) + 1
        else:
            current.append(sent)
            length += len(sent) + 1
    if current:
        chunks.append(' '.join(current))
    return chunks   # returns [] on empty text — no [''] sentinel

In [25]:
# ── Normalisation config ───────────────────────────────────────────────────────
ABBREV = {
    r'\bMo\.?\b':   'Missouri',
    r'\bKan\.?\b':  'Kansas',
    r'\bKas\.?\b':  'Kansas',
    r'\bKs\.?\b':   'Kansas',
    r'\bNeb\.?\b':  'Nebraska',
    r'\bIll\.?\b':  'Illinois',
    r'\bCal\.?\b':  'California',
    r'\bMass\.?\b': 'Massachusetts',
    r'\bPa\.?\b':   'Pennsylvania',
    r'\bN\.Y\.?\b': 'New York',
    r'\bD\.C\.?\b': 'Washington D.C.',
}

ENTITY_REMAP = {
    ('GPE', 'Providence Medical Center'): ('ORG', 'Providence Medical Center'),
    ('GPE', 'providence-health'):         ('ORG', 'Providence Health'),
    ('ORG', 'providence-health'): ('ORG', 'Providence Health'),
}

MEDICAL_SUFFIXES = re.compile(r'\b(M\.D\.?|Dr\.?|Jr\.?|Sr\.?|Ph\.D\.?|Esq\.?|M$)', re.I)


# ── Normalisation helpers ──────────────────────────────────────────────────────
def normalise_entity(name: str, category: str) -> str:
    name = re.sub(r'^[ĠĊ\s]+|[ĠĊ\s]+$', '', name)
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'[,;:]+$', '', name).strip()

    if category == 'GPE':
        name = name.rstrip('.')
        for pattern, expansion in ABBREV.items():
            name = re.sub(pattern, expansion, name)
        name = name.strip()

    if category == 'GPE':
        # Drop highway/road identifiers
        if re.fullmatch(r'(I|US|Hwy|Rt|Route|SR)-?\d+', name, re.I):
            return ''
            
    if len(name) <= 1:
        return ''
    if re.fullmatch(r'[A-Z]\.?', name):
        return ''
    if category == 'PERSON' and MEDICAL_SUFFIXES.fullmatch(name):
        return ''

    return name

def apply_remap(category: str, name: str) -> tuple[str, str]:
    remapped = ENTITY_REMAP.get((category, name.lower()))
    if remapped:
        return remapped
    return (category, name)

def clean_entity_name(name: str) -> str:
    """Legacy shim — kept so callers don't break."""
    return name.strip()

# ── Main processing loop ───────────────────────────────────────────────────────
def process_dataset(root_path, file_limit=None):
    files = get_file_list(root_path, limit=file_limit)
    counts = {'PERSON': Counter(), 'GPE': Counter(), 'ORG': Counter()}

    print(f'Running NER on {len(files):,} files...')

    all_chunks = []
    for fp in files:
        text = extract_text(fp)
        all_chunks.extend(chunk_text(text))
    all_chunks = [c for c in all_chunks if c and c.strip()]

    print(f'  → {len(all_chunks):,} text chunks to process')

    if not all_chunks:
        raise ValueError(f"No text chunks produced from '{root_path}'.")

    for batch_result in tqdm(
        ner_pipe(all_chunks, batch_size=BATCH_SIZE),
        total=len(all_chunks),
        desc='NER progress',
        unit='chunk',
    ):
        for entity in batch_result:
            category = map_label(entity['entity_group'])
            if not category:
                continue
            name = normalise_entity(entity['word'], category)
            if name:
                category, name = apply_remap(category, name)
                counts[category][name] += 1

    # ── Merge prefix duplicates ────────────────────────────────────────────────
    for cat in ('GPE', 'ORG', 'PERSON'):
        counter = counts[cat]
        keys = sorted(counter.keys(), key=len, reverse=True)
        for i, long_key in enumerate(keys):
            for short_key in keys[i+1:]:
                if long_key.startswith(short_key) and counter[short_key]:
                    counter[long_key] += counter.pop(short_key)

    # ── Merge bare surnames into full names ────────────────────────────────────
    person_counter = counts['PERSON']
    full_names = {k for k in person_counter if ' ' in k}
    for surname in list(person_counter):
        if ' ' not in surname:
            matching = [f for f in full_names if f.split()[-1] == surname]
            if len(matching) == 1:
                person_counter[matching[0]] += person_counter.pop(surname)

    return counts

In [26]:
# ── Run ───────────────────────────────────────────────────────────────────────
# Set file_limit=None to process all ~100k files.

results = process_dataset(TARGET_DIR, file_limit=None)

# Add this temporarily after process_dataset returns:
for cat, counter in results.items():
    for name in counter:
        if 'Providence' in name:
            print(cat, name, counter[name])

# ── Export ────────────────────────────────────────────────────────────────────
all_data = [
    {'Category': cat, 'Entity': name, 'Frequency': freq}
    for cat, counter in results.items()
    for name, freq in counter.most_common()
]

df_results = pd.DataFrame(all_data)
df_results.to_csv('ner_results_summary.csv', index=False)
print("Results exported to 'ner_results_summary.csv'")
print(df_results.sort_values('Frequency', ascending=False).head(15).to_string(index=False))

Scanning directory for files...
Running NER on 73,041 files...
  → 481,036 text chunks to process


NER progress: 100%|██████████████████████████████████████████████████████| 481036/481036 [00:45<00:00, 10687.65chunk/s]


GPE Providence Medical center 422
GPE Providence Baptist Church 57
GPE Providence Park 2
GPE Wheatley Providence 2
GPE Providence St. Margaret Medical center 12
GPE Providence Place Nursing Facility 23
GPE Providence Missionary Baptist Church 96
GPE Providence Hospital 13
GPE Centerpoint and Providence 3
GPE Bethany and Providence Medical center 3
GPE Providence hospital 12
GPE Lake Providence 11
GPE South Providence Medical Building 8
GPE Providence St. Margaret Hospital 3
GPE Margaret’s Providence Hospital 12
GPE �s Providence Hospital 1
GPE Providence and Saint John Emergency Services 2
GPE Providence Ball 2
GPE Providence Family Care Center 1
GPE Log Providence Baptist 3
GPE Providence Medical Center Family Care Center 129
GPE Providence-St. Margaret Hospital 9
GPE Providence Rd 3
GPE The Providence Medical center 3
GPE Providence Saint Margaret 3
GPE Providence Road 2
GPE Providence Saint John Medical Center of Kansas 115
GPE Old Providence Road 2
GPE Providence Presbyterian 2
GPE

In [ ]:
# ── Quick spot-check on a single text ─────────────────────────────────────────
sample = 'President Truman met with General MacArthur in Kansas City, Missouri, on behalf of the United Nations.'
for ent in ner_pipe(sample):
    print(f"{ent['entity_group']:6s}  {ent['word']:30s}  score={ent['score']:.3f}")